# π 脉冲幅度标定

参数在下一单元设置；只有打开 RUN_HARDWARE 才连接并提交实验。扫描返回 IQ 距离最大的**候选幅度**。复测并确认后再发布配置。

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == "execute":
    ROOT = ROOT.parent
if not (ROOT / "src").is_dir():
    raise FileNotFoundError("Start Jupyter in Platform_LQ or its execute directory")
sys.path.insert(0, str(ROOT))
import numpy as np
from src.pi import PulseSettings, run_calibration, plot_qubit, waveform_budget
from src.config import save_hardware, publish_pi
BACKEND_NAME = "YOUR_BACKEND"
QUBITS = [9]
AMPLITUDES = {9: np.linspace(3.5, 4.5, 21)}
DRIVE_IF_GHZ = {}
SHOTS = 1000
SETTINGS = PulseSettings()
RUN_HARDWARE = False
RUN_MODE = "new"  # "new" or "resume"
RESUME_RUN_TAG = ""
PUBLISH_CONFIG = False  # Set only after independent validation


In [ ]:
run = None
if RUN_HARDWARE:
    from src.progress import watch_run
    with watch_run(ROOT / "result" / "pi", sum(1 + len(AMPLITUDES[q]) for q in QUBITS),
                   run_mode=RUN_MODE, resume_run_tag=RESUME_RUN_TAG,
                   title='π 振幅网格标定'):
        from lqcloud import LQCloudProvider
        backend = LQCloudProvider().get_backend(BACKEND_NAME)
        qconf = {int(q["id"]): q for q in backend.config["properties"]["qubit_metrics"]}
        print("Planned jobs:", sum(1 + len(AMPLITUDES[q]) for q in QUBITS))
        for q in QUBITS:
            print("Q", q, waveform_budget(qconf[q], SETTINGS))
        hardware_file = save_hardware(ROOT / "config", backend)
        run, candidates = run_calibration(
            backend, qconf, {q: AMPLITUDES[q] for q in QUBITS},
            ROOT / "result" / "pi", settings=SETTINGS, drive_ifs=DRIVE_IF_GHZ,
            shots=SHOTS, run_mode=RUN_MODE, resume_run_tag=RESUME_RUN_TAG)
        print(run, candidates)


In [ ]:
# To inspect a completed earlier run, assign its absolute path to run first.
if run is not None:
    import matplotlib.pyplot as plt
    for q in QUBITS:
        plot_qubit(run, q)
    plt.show()
if PUBLISH_CONFIG:
    if run is None:
        raise ValueError("Select a completed run before publishing")
    print("Published:", publish_pi(run, ROOT / "config"))
